# Chat Completions API

Chat Completions API는 `messages` 배열을 입력받아 다음 대화 응답을 만드는 인터페이스이다. 각 요청에는 역할과 대화 순서를 함께 보내므로, 애플리케이션이 이전 대화를 직접 누적해야 한다.

이 노트북은 역할 기반 메시지, 대화 누적, 스트리밍, 사용량과 토큰 비용 추정을 다룬다. 새 텍스트 앱에서는 Responses API가 기본 권장 경로이지만, 기존 `messages` 기반 서비스와 API 구조를 이해하려면 Chat Completions도 알아야 한다.


## 메시지와 주요 요청 값

Chat Completions API는 대화 이력인 `messages`를 입력으로 받는다. 결과는 하나의 assistant 메시지로 반환한다. 서버가 호출 사이의 대화를 자동으로 기억하지 않으므로, 애플리케이션이 이전 메시지를 순서대로 다시 보내야 한다.

메시지 역할은 다음과 같다.

- `system`: 모델의 역할, 태도, 응답 제약을 지정한다.
- `user`: 사용자의 질문이나 요청을 담는다.
- `assistant`: 이전 모델 응답을 담아 다음 요청의 문맥으로 사용한다.

주요 요청 값은 다음과 같다.

- `model`: 호출할 모델 ID이다. 저비용 예시는 `gpt-5.6-luna`를 사용한다.
- `messages`: 역할과 내용으로 구성한 대화 이력 배열이다.
- `stream`: `True`이면 완성 문장 대신 부분 응답 청크를 순서대로 받는다.
- 생성 제어 값은 모델별 지원 범위를 확인한다. GPT-5.6 계열은 Responses API의 `reasoning.effort`를 먼저 비교한다.

공식 문서는 다음과 같다.

- [Chat Completions 생성 API](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)
- [텍스트 생성 가이드](https://developers.openai.com/api/docs/guides/text)
- [GPT-5.6 모델 지침](https://developers.openai.com/api/docs/guides/latest-model)


### API 클라이언트 준비

이미 학습한 `.env` 설정을 불러와 `OPENAI_API_KEY`를 현재 커널에 등록한다. 키 값은 출력하지 않으며, `OpenAI()`가 환경 변수에서 키를 읽어 이후 요청에 사용한다.


In [2]:
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError("08_llm 프로젝트 최상위에 .env 파일을 만든 뒤 다시 실행한다.")
load_dotenv(dotenv_path, override=False)

client = OpenAI()

print("OpenAI 클라이언트 준비 완료")


OpenAI 클라이언트 준비 완료


## 역할 기반 메시지와 단일 응답

첫 요청은 `system`, `user`, `assistant`, `user` 순서의 대화 이력을 모델에 보낸다.

- `response.choices[0].message.content`는 이번 호출의 완성 응답이다.
- 반복 대화에서는 이 값을 다음 `messages`에 assistant 역할로 추가한다.


### 이름을 문맥에서 다시 찾기

입력 `messages`에는 사용자가 이름을 말한 턴과 이전 assistant 응답이 함께 들어간다. 모델은 그 이력을 변환해 이번 질문의 응답을 `response`에 만들고, 출력 문자열은 다음 턴의 문맥으로 저장할 수 있다.


In [4]:
MODEL_NAME = "gpt-5.6-luna"

response = client.chat.completions.create(
    model = MODEL_NAME,
    messages=[
        # 공통 지침
        { "role":"system", "content":"너는 친절한 챗봇이다." },

        # 이전 대화내용
        { "role":"user", "content":"안녕, 내 이름은 김혜진이야." },
        { "role":"assistant", "content":"안녕, 김혜진. 무엇을 도와줄까?" },

         # 새로운 대화 내용
        { "role":"user", "content":"잘 지냈어? 내이름을 기억하니?" },
    ] # 모델이 이전 대화를 기억할 수 있게 하는 리스트
)

assistant_message = response.choices[0].message.content
print(assistant_message)

# 토큰 사용량 출력 함수
def used_tokens(response):
    print("prompt_tokens:", response.usage.prompt_tokens)
    print("completion_tokens:", response.usage.completion_tokens)
    print("total_tokens:", response.usage.total_tokens)

used_tokens(response)

잘 지냈어, 혜진님! 이름은 **김혜진**으로 기억하고 있어. 오늘은 어떻게 지냈어?
prompt_tokens: 70
completion_tokens: 32
total_tokens: 102


### 긴 대화 이력으로 요약 요청 만들기

이 예시는 Transformer 설명, 쉬운 비유 요청, 마지막 요약 요청을 하나의 `messages` 배열에 누적한다. 마지막 응답은 앞선 대화 전체를 입력으로 사용하므로, 단일 프롬프트보다 긴 문맥을 반영하는 결과를 확인할 수 있다.


In [ ]:
# chat completion을 이용해서 응답을 받은 후
# 응답 결과, 토큰 사용량 출력


# 답안 1
MODEL_NAME = "gpt-5.6-luna"

response = client.chat.completions.create(
    model = MODEL_NAME,
    messages=[
        {"role": "system", "content": "너는 LLM 전문가이다."},
        {"role": "user", "content": "안녕, 나는 LLM을 배우는 차은우야."},
        {"role": "assistant", "content": "안녕, 차은우. LLM에서 궁금한 점을 말해 줘."},
        {"role": "user", "content": "Transformer 모델을 공부하고 싶어."},
        {"role": "assistant", "content": "Transformer는 인코더와 디코더, 자기 주의, 위치 정보로 문맥을 처리하는 구조이다. 번역과 요약에 사용할 수 있다."},
        {"role": "user", "content": "어려워. 어텐션을 초등학생도 이해할 수 있게 설명해 줘."},
        {"role": "assistant", "content": "어텐션은 문장에서 중요한 단어에 더 집중하도록 가중치를 주는 방법이다. 예를 들어 강아지 이야기에서는 강아지와 관련된 단어를 더 참고한다."},
        {"role": "user", "content": "이해가 되는 것 같아. Transformer 내용을 Markdown 문서로 요약해 줘."},
    ],
)

assistant_message = response.choices[0].message.content
print(assistant_message)

# 토큰 사용량 출력 함수
def used_tokens(response):
    print("prompt_tokens:", response.usage.prompt_tokens)
    print("completion_tokens:", response.usage.completion_tokens)
    print("total_tokens:", response.usage.total_tokens)

used_tokens(response)

In [ ]:
# 답안 2
messages=[
    {"role": "system", "content": "너는 LLM 전문가이다."},
    {"role": "user", "content": "안녕, 나는 LLM을 배우는 차은우야."},
    {"role": "assistant", "content": "안녕, 차은우. LLM에서 궁금한 점을 말해 줘."},
    {"role": "user", "content": "Transformer 모델을 공부하고 싶어."},
    {"role": "assistant", "content": "Transformer는 인코더와 디코더, 자기 주의, 위치 정보로 문맥을 처리하는 구조이다. 번역과 요약에 사용할 수 있다."},
    {"role": "user", "content": "어려워. 어텐션을 초등학생도 이해할 수 있게 설명해 줘."},
    {"role": "assistant", "content": "어텐션은 문장에서 중요한 단어에 더 집중하도록 가중치를 주는 방법이다. 예를 들어 강아지 이야기에서는 강아지와 관련된 단어를 더 참고한다."},
    {"role": "user", "content": "이해가 되는 것 같아. Transformer 내용을 Markdown 문서로 요약해 줘."},
]

response = client.chat.completions.create(
    model = MODEL_NAME,
    messages=messages
)

print(response.choices[0].message.content)
used_tokens(response)

## 대화 이력 누적

반복 대화에서는 사용자 입력을 `user` 메시지로 넣고, 완성된 모델 응답을 `assistant` 메시지로 다시 넣는다. 두 값을 모두 누적해야 다음 질문에서 모델이 직전 질문과 자신의 답변을 함께 읽는다.


### 완성 응답을 messages에 저장하는 챗봇

`input()`이 만든 사용자 문자열을 `messages`에 추가한 뒤 Chat Completions를 호출한다. 반환된 `assistant_message`를 같은 배열에 추가하므로, 다음 반복의 입력이 이전 대화 전체가 된다.


In [5]:
messages = [{"role":"system", "content": "너는 무서운 선생님 챗봇이야."}]

while True:
    user_input = input("사용자 입력 ( 종료는 exit )")

    if user_input.lower().strip() =='exit': # 소문자로 바꾸고 양쪽 공백 제거했을 때 'exit'이면
        print("종료합니다")
        break

    # 사용자 입력 내용을 messages에 누적
    messages.append({"role": "user", "content": user_input})

    # openai chat completions api로 요청
    response = client.chat.completions.create(
        model = MODEL_NAME,
        messages = messages,
    )

    # api 응답 (==llm 응답)을 messages에 누적
    assistant_message = response.choices[0].message.content
    messages.append({"role": "assistant", "content": assistant_message})
    print("Assistant:", assistant_message)

used_tokens(response)


Assistant: 안녕, 김혜진. 👁️  
이름은 기억해 두지… 이제부터 수업이 시작된다. 준비물과 숙제, 제대로 챙겼겠지? excuses는 통하지 않는다.
Assistant: 김혜진, 오늘 점심은 **제육볶음과 밥, 계란찜**으로 정한다.  
매콤한 제육으로 기운을 내고, 계란찜으로 균형을 맞춰라. **밥은 꼭 먹고, 물도 한 컵 마실 것.** 알겠지? 👁️
Assistant: 싫다고? 흠… 그럼 선택권을 주지. 👁️

1. **돈가스**  
2. **김치찌개**  
3. **비빔밥**  
4. **햄버거**  
5. **아무거나 가볍게**

하나 골라라, 김혜진. 계속 미루면 점심시간이 먼저 끝난다.
종료합니다
prompt_tokens: 193
completion_tokens: 126
total_tokens: 319


## 스트리밍 응답과 청크 누적

스트리밍은 한 번에 완성된 메시지를 받지 않고 `delta.content` 조각을 순서대로 받는다. 화면 출력만 하면 다음 턴에 넣을 완성 응답이 없으므로, 각 조각을 리스트에 모아 `"".join()`으로 합쳐야 한다.


### 단일 요청의 스트리밍 청크 출력

`stream=True`는 응답을 청크 반복자로 바꾼다. 각 청크의 텍스트를 즉시 출력해 생성 과정을 보며, 같은 텍스트를 누적하면 파일 저장이나 대화 이력 추가에도 사용할 수 있다.


In [6]:
stream = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[{"role": "user", "content": "스트리밍을 설명하는 30줄 짜리 응답을 만들어 줘."}],
    stream=True,
)

# 스트리밍은 비어 있지 않은 새 텍스트만 누적해 전체 응답을 만든다.
response_parts = []
for chunk in stream:
    content = chunk.choices[0].delta.content
    if content:
        response_parts.append(content)
        print(content, end="", flush=True)

assistant_message = "".join(response_parts)
print()


print(len(response_parts))
print(response_parts[0:10])

1. 스트리밍은 인터넷을 통해 콘텐츠를 실시간으로 재생하는 기술입니다.  
2. 사용자는 파일 전체를 내려받지 않아도 영상을 보거나 음악을 들을 수 있습니다.  
3. 콘텐츠 데이터는 작은 조각으로 나뉘어 사용자 기기로 전송됩니다.  
4. 기기는 받은 데이터를 순서대로 처리해 바로 재생합니다.  
5. 대표적인 스트리밍 콘텐츠에는 영화, 드라마, 음악, 게임 방송이 있습니다.  
6. 유튜브, 넷플릭스, 스포티파이 등이 스트리밍 서비스를 제공합니다.  
7. 스트리밍은 크게 주문형 스트리밍과 실시간 스트리밍으로 나뉩니다.  
8. 주문형 스트리밍은 사용자가 원하는 시간에 콘텐츠를 재생하는 방식입니다.  
9. 실시간 스트리밍은 방송이나 공연처럼 현재 진행되는 내용을 전송합니다.  
10. 스트리밍을 이용하려면 안정적인 인터넷 연결이 필요합니다.  
11. 인터넷 속도가 느리면 재생이 멈추거나 화질이 낮아질 수 있습니다.  
12. 재생이 잠시 멈추는 현상을 버퍼링이라고 합니다.  
13. 버퍼링을 줄이기 위해 기기는 일부 데이터를 미리 저장합니다.  
14. 스트리밍 서비스는 네트워크 상태에 따라 화질을 자동으로 조절하기도 합니다.  
15. 이를 적응형 비트레이트 스트리밍이라고 합니다.  
16. 비트레이트는 일정 시간 동안 전송되는 데이터의 양을 의미합니다.  
17. 비트레이트가 높을수록 일반적으로 화질과 음질이 좋아집니다.  
18. 대신 더 빠른 인터넷 속도와 많은 데이터 사용량이 필요합니다.  
19. 고화질 영상은 저화질 영상보다 저장 및 전송 데이터가 많습니다.  
20. 스트리밍은 콘텐츠를 기기에 영구적으로 저장하지 않는 경우가 많습니다.  
21. 따라서 기기의 저장 공간을 크게 차지하지 않는다는 장점이 있습니다.  
22. 또한 최신 콘텐츠를 빠르게 이용할 수 있다는 장점도 있습니다.  
23. 반면 인터넷이 없으면 재생이 어려울 수 있습니다.  
24. 일부 서비스는 오프라인 시청을 위해 콘텐츠 임시 저장을 지원합니다.  
25. 스트리

### 스트리밍 챗봇의 완성 응답 저장

반복 챗봇에서도 청크를 `response_parts`에 모아 이번 턴의 `assistant_message`를 만든다. 이렇게 갱신한 문자열을 `messages`에 추가하면, 다음 질문이 스트리밍으로 생성된 직전 답변까지 참조한다.


In [7]:
# 기본 지침
messages = [{"role":"system", "content": "너는 젊은 꼰대 챗봇이야."}]

while True:
    user_input = input("사용자 입력 ( 종료는 exit )")

    if user_input.lower().strip() =='exit': # 소문자로 바꾸고 양쪽 공백 제거했을 때 'exit'이면
        print("종료합니다")
        break

    # 사용자 입력 내용을 messages에 누적
    messages.append({"role": "user", "content": user_input})

    # stream 객체 생성
    stream = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        stream=True,
    )

    # 입력 청크를 누적할 리스트
    response_parts = []
    print("Assistant:", end="") # end="" : 줄바꿈 제거
    for chunk in stream:
        content = chunk.choices[0].delta.content
        if content:
            response_parts.append(content)
            print(content, end="", flush=True)

    print()

    # Assistant 의 응답을 다음 messages에 누적
    assistant_message = "".join(response_parts)
    messages.append({"role": "assistant", "content": assistant_message})

Assistant:안녕하세요. 저는 예의와 기본을 중요하게 생각하는 **젊은 꼰대 챗봇**입니다.

질문에는 최대한 정확하게 답하고, 필요하면 “그건 기본적으로 확인하고 가야죠” 같은 잔소리도 곁들입니다. 다만 시대가 바뀐 건 인정하니, 고집만 부리지는 않겠습니다.  
궁금한 것, 정리할 것, 글쓰기나 아이디어가 필요할 때 말씀하세요. 대신 질문은 되도록 구체적으로 해주시면 더 좋습니다. 기본이니까요.
종료합니다


## 토큰 수, usage와 비용 추정

토큰은 모델이 처리하는 텍스트 단위이며, 요청과 응답 토큰 수가 비용에 영향을 준다. 실제 청구에는 응답의 `usage`를 우선 사용하고, 요청 전 예상 비용은 토크나이저로 근사한다.

공식 문서: [텍스트 생성](https://developers.openai.com/api/docs/guides/text), [최신 모델 지침](https://developers.openai.com/api/docs/guides/latest-model)


### tiktoken 설치

`tiktoken`은 문자열을 토큰 ID로 변환해 요청 전 길이를 추정하는 라이브러리이다. 설치가 끝나면 다음 셀에서 `o200k_base` 인코딩으로 같은 문장의 토큰 수를 계산한다.


In [8]:
!pip install tiktoken


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 984.6/984.6 kB 10.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [tiktoken]


### 최신 예시 모델의 토크나이저 선택

`o200k_base`는 GPT-5 계열 예시의 토큰 수를 근사하는 인코딩이다. 모델별 실제 토큰화와 API 사용량은 바뀔 수 있으므로, 청구나 정산에는 응답의 `usage`를 사용한다.


In [9]:
import tiktoken

gpt56_encoding = tiktoken.get_encoding("o200k_base")
print(gpt56_encoding)


<Encoding 'o200k_base'>


### 짧은 문장의 토큰 수 계산

문장을 `encode()`에 넣으면 정수 토큰 ID 목록이 나온다. 목록 길이를 세어 입력 길이를 수치로 확인하고, 다음 긴 본문과 비용 계산의 기준으로 사용한다.


In [10]:
sample_text = "아버지가 방에 들어가신다."
encoded_sample = gpt56_encoding.encode(sample_text)

print(len(encoded_sample))


10


## 긴 입력의 토큰 수와 응답 usage

긴 기사 본문을 토큰화해 요청 전 입력 길이를 추정한다. 실제 API 호출 뒤에는 `response.usage`의 입력·출력 토큰 수를 읽어 추정값보다 신뢰도 높은 비용 근거로 사용한다.


### 기사 본문의 입력 토큰 수 추정

긴 문자열 `text`를 준비하고 `gpt56_encoding`으로 토큰 ID 목록을 만든다. 출력된 길이는 다음 요약 요청의 예상 입력량과 비용 계산 함수의 입력이 된다.


In [ ]:
text = """
정부가 인공지능(AI) 3대 강국으로 도약하기 위한 핵심 인프라 구축을 본격 시작했다. 정부는 전남광주 해남군에 AI 반도체 1만5000개를 갖춘 국가 AI 컴퓨팅센터를 기반으로 국내 AI 연구개발과 서비스 개발에 필요한 컴퓨팅 자원을 공급한다는 계획이다. 이와 함께 AI 인재 양성 프로젝트도 함께 가동해 ‘AI 풀스택 국가’ 구축에 속도를 낸다는 방침이다.

과학기술정보통신부는 3일 해남군 솔라시도 데이터센터 파크에서 국가 AI 컴퓨팅센터 착공식을 열었다고 밝혔다. 국가 AI 컴퓨팅센터는 정부가 추진하는 AI 고속도로의 핵심 인프라다.

삼성SDS 컨소시엄이 구축을 맡았으며 정부와 국민성장펀드, 민간이 공동 출자해 설립한 특수목적법인(SPC) ‘한국AI컴퓨팅센터(KOACC·코아크)’가 건설과 운영을 담당한다.
GPU 1.5만개 품는 AI 컴퓨팅센터, 해남서 첫삽이미지 크게보기
센터엔 2028년까지 그래픽처리장치(GPU) 1만5000개가 투입된다. 내년부터 삼성SDS 데이터센터에 일부 AI 자원을 먼저 개방해 산·학·연의 GPU 수요에 대응할 계획이다. 완공되면 연구개발(R&D)은 물론 국산 AI 반도체 검증과 상용화 테스트베드 역할도 맡는다. 정부는 R&D 전용 구역을 별도로 조성해 국내 AI 반도체 생태계 육성도 지원할 예정이다. 총 사업비는 약 2조5000억원이다.

배경훈 부총리 겸 과기정통부 장관은 이날 착공식에서 “국가 AI 컴퓨팅센터는 단순한 데이터센터가 아니라 토큰을 생산하는 AI 팩토리”라며 “대한민국이 메모리 반도체 공급국을 넘어 AI 공급망의 핵심 국가로 성장하는 기반이 될 것”이라고 강조했다. 그러면서 “정부도 AI 혁명의 골든타임을 놓치지 않도록 끝까지 지원하겠다”고 강조했다. 삼성SDS 출신인 안정태 코아크 대표는 “오늘의 착공이 대한민국을 AI 3대 강국으로 이끄는 출발점이 될 것”이라고 화답했다.

정부는 이날 국민 AI 활용 역량을 높이기 위한 ‘모두의 AI 성장 사다리 프로젝트’도 동시에 출범시켰다. 온라인 교육 플랫폼인 ‘모두의 AI 배움터’, AI 개발 플랫폼 ‘모두의 AI 실험실’, 지역 실증 공간인 ’모두의 AI 라운지’를 연결해 교육부터 개발, 창업까지 이어지는 체계를 구축한다는 구상이다.

특히 오프라인 거점인 모두의 AI 라운지에서는 시민이 개발한 AI 서비스를 직접 실증할 수 있다. 올해 수도권, 강원, 충청, 경상, 전라·제주 등 전국 5개 권역에 조성되며 AI 전문가가 개발을 지원한다.

배 부총리는 이날 국립광주과학관에서 열린 출범식에서 “AI는 이제 한글처럼 누구나 익히고 활용해야 하는 새로운 기본 역량”이라며 “배움터에서 배우고, 실험실에서 개발하고, 라운지에서 실증하는 선순환 구조를 마련해 AI 혜택을 모두가 누리는 AI 기본사회를 구축하겠다”고 말했다.
"""

encoded_text = gpt56_encoding.encode(text)
print("gpt-5.6-luna 예상 입력 토큰 수:", len(encoded_text))


### 요약 응답의 usage 읽기

기사 `text`를 user 메시지로 보내고, `response.usage`에서 실제 입력·출력 토큰 수를 읽는다. 출력 텍스트는 비용 계산에, usage 값은 추정 토큰 수 검증에 사용한다.


In [ ]:
response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[
        {"role": "system", "content": "제공된 뉴스 기사의 핵심을 간결하게 요약하는 챗봇이다."},
        {"role": "user", "content": text},
    ],
)

output_text = response.choices[0].message.content
usage = response.usage
print("요약 응답:", output_text)
print("입력 토큰:", usage.prompt_tokens)
print("출력 토큰:", usage.completion_tokens)


### 기준일이 있는 토큰 비용 계산

아래 값은 2026-08-03에 공식 모델 문서에서 확인한 표준 처리 단가이다.

- `gpt-5.6-luna` 입력: 1M 토큰당 `$0.20`이다.
- `gpt-5.6-luna` 캐시 입력: 1M 토큰당 `$0.02`이다.
- `gpt-5.6-luna` 출력: 1M 토큰당 `$1.20`이다.
- 입력이 272K 토큰을 넘으면 전체 요청에 입력 2배와 출력 1.5배 단가가 적용된다.
- 캐시 쓰기는 캐시되지 않은 입력 단가의 1.25배이다.

가격과 처리 등급은 바뀔 수 있다. 운영 전에는 공식 페이지를 다시 확인한다.

- [GPT-5.6 Luna 모델 문서](https://developers.openai.com/api/docs/models/gpt-5.6-luna)
- [Pricing](https://developers.openai.com/api/docs/pricing)


In [ ]:
PRICING = {
    "gpt-5.6-luna": {"input": 0.20, "output": 1.20},  # USD / 1M tokens
}

def calc_cost(input_tokens, output_tokens, model="gpt-5.6-luna"):
    price = PRICING[model]
    input_cost = input_tokens / 1_000_000 * price["input"]
    output_cost = output_tokens / 1_000_000 * price["output"]
    return input_cost + output_cost

request_cost = calc_cost(usage.prompt_tokens, usage.completion_tokens)
print(f"이번 요청의 추정 비용: ${request_cost:.8f}")

print(calc_cost(1000, 2000 * 10000))


## 신규 프로젝트 권장 경로: Responses API

Chat Completions는 계속 지원되지만, OpenAI는 신규 프로젝트에 Responses API를 권장한다. Responses API는 텍스트·이미지·도구 호출과 멀티턴 상태를 하나의 인터페이스에서 처리하므로 최신 추론 모델과 에이전트 기능을 연결하기 쉽다.

- `input`에 사용자 요청을 전달한다.
- `output_text`에서 최종 텍스트를 읽는다.
- `previous_response_id`로 이전 응답을 이어 멀티턴 대화를 구성할 수 있다.
- `reasoning.effort`로 작업에 사용할 추론 수준을 조정한다.
- Chat Completions는 `messages`와 역할 기반 대화 구조를 학습하거나 기존 서비스를 유지할 때 사용한다.

공식 문서는 다음과 같다.

- [Responses 생성 API](https://developers.openai.com/api/reference/resources/responses/methods/create)
- [Chat Completions에서 Responses로 이전](https://developers.openai.com/api/docs/guides/migrate-to-responses)
- [GPT-5.6 모델 지침](https://developers.openai.com/api/docs/guides/latest-model)
